# Online Shoppers — EDA & Purchase Prediction

Analyses web session data to identify what drives conversions, and trains a logistic regression classifier to predict purchases.

**Dataset:** [Online Shoppers Purchasing Intention](https://www.kaggle.com/datasets/imakash3011/online-shoppers-purchasing-intention-dataset) — place the file at `data/online_shoppers_intention.csv`

## 1. Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('data/online_shoppers_intention.csv')
print(df.shape)
print(df.head())
print(f"\nConversion rate: {df['Revenue'].mean()*100:.1f}%")

## 2. Traffic Source Analysis

In [ ]:
# Conversion rate and volume per traffic type
source_revenue = df.groupby('TrafficType').agg(
    ConversionRate=('Revenue', 'mean'),
    TotalVisits=('Revenue', 'count')
).reset_index()

source_revenue['ConversionRate'] = source_revenue['ConversionRate'] * 100
source_revenue['NotConverted'] = 100 - source_revenue['ConversionRate']
source_revenue['TrafficType'] = source_revenue['TrafficType'].astype(str)
source_revenue = source_revenue.sort_values('ConversionRate', ascending=True)
source_revenue

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Left: conversion rate per traffic type
ax1.barh(source_revenue['TrafficType'], source_revenue['ConversionRate'], color='steelblue')
ax1.set_yticks(range(len(source_revenue)))
ax1.set_yticklabels(source_revenue['TrafficType'])
ax1.set_xlabel('Conversion Rate (%)')
ax1.set_title('Conversion Rate by Traffic Type')

# Right: total visits per traffic type
ax2.barh(source_revenue['TrafficType'], source_revenue['TotalVisits'], color='orange')
ax2.set_yticks(range(len(source_revenue)))
ax2.set_yticklabels(source_revenue['TrafficType'])
ax2.set_xlabel('Total Visits')
ax2.set_title('Total Visits by Traffic Type')

plt.tight_layout()
plt.show()

## 3. Visitor Type Analysis

In [ ]:
visitor_stats = df.groupby('VisitorType').agg(
    ConversionRate=('Revenue', 'mean'),
    TotalVisits=('Revenue', 'count')
).reset_index()

visitor_stats['ConversionRate'] = visitor_stats['ConversionRate'] * 100
visitor_stats = visitor_stats.sort_values('ConversionRate', ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(visitor_stats['VisitorType'], visitor_stats['ConversionRate'])
plt.xlabel('Conversion Rate (%)')
plt.title('Conversion Rate by Visitor Type')
plt.tight_layout()
plt.show()

## 4. Bounce Rate by Month

In [ ]:
bounce_rate = df.groupby('Month').agg({'BounceRates': 'mean'}).reset_index()
bounce_rate['BounceRate_pct'] = bounce_rate['BounceRates'] * 100

plt.figure(figsize=(10, 4))
plt.bar(bounce_rate['Month'], bounce_rate['BounceRate_pct'], color='coral')
plt.xlabel('Month')
plt.ylabel('Avg Bounce Rate (%)')
plt.title('Average Bounce Rate by Month')
plt.tight_layout()
plt.show()

## 5. Conversion Funnel

In [ ]:
step1 = len(df)
step2 = len(df[df['ProductRelated'] > 0])
step3 = len(df[df['ProductRelated_Duration'] > 0])
step4 = len(df[df['Revenue'] == True])

steps = [step1, step2, step3, step4]
labels = ['All visits', 'Viewed listing', 'Engaged', 'Converted']

plt.figure(figsize=(10, 5))
bars = plt.barh(labels, steps, color=['#1f77b4', '#4a90d9', '#7fb3e8', '#b3d4f5'])
for i, v in enumerate(steps):
    plt.text(v, i, f' {v:,}', va='center')
plt.gca().invert_yaxis()
plt.xlabel('Number of Sessions')
plt.title('Conversion Funnel')
plt.tight_layout()
plt.show()

## 6. Feature Correlation with Revenue

In [ ]:
df_numeric = df.select_dtypes(include=['int64', 'float64', 'bool'])
correlation = df_numeric.corr()['Revenue'].sort_values(ascending=False)
print(correlation)

## 7. Purchase Prediction — Logistic Regression

In [ ]:
# Encode categorical columns
le = LabelEncoder()
df['Month_encoded'] = le.fit_transform(df['Month'])
df['VisitorType_encoded'] = le.fit_transform(df['VisitorType'])

features = ['PageValues', 'ProductRelated', 'ProductRelated_Duration',
            'Administrative', 'BounceRates', 'ExitRates',
            'Informational', 'Administrative_Duration',
            'SpecialDay', 'Month_encoded', 'VisitorType_encoded']

X = df[features]
y = df['Revenue']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# class_weight='balanced' compensates for fewer purchase sessions
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))